<a href="https://colab.research.google.com/github/Gaurang983/rsp-warehouse-inspection/blob/index.html/q1120205_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q pybullet matplotlib numpy


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 MB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
import json, math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pybullet as p
import pybullet_data
assets = Path("warehouse_assets")
assets.mkdir(exist_ok=True)
if p.isConnected(): p.disconnect()
client = p.connect(p.DIRECT)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.setGravity(0, 0, -9.81)
p.loadURDF("plane.urdf")



0

In [ ]:

def box_model(name, size, color):
    # Static semantic objects share a simple geometry generator but distinct models.
    xyz = " ".join(map(str, size))
    rgba = " ".join(map(str, color))
    text = f'''




'''
    path = assets / f"{name}.urdf"
    path.write_text(text)
    return str(path)
models = {
    "barrier": box_model("barrier", [0.18, 2.0, 0.7], [0.6, 0.63, 0.7, 1]),
    "shelf": box_model("shelf", [0.7, 1.0, 0.9], [0.8, 0.45, 0.15, 1]),
    "inspection_zone": box_model("inspection_zone", [0.65, 0.65, 0.02], [0.15, 0.8, 0.3, 1]),
    "landmark": box_model("landmark", [0.2, 0.2, 1.0], [0.2, 0.35, 1, 1]),
}
# Two driven wheel joints and a passive support sphere.
robot_xml = '''


'''
for side, y in [("left", 0.18), ("right", -0.18)]:
    robot_xml += f'''


'''
robot_xml += '''
'''
(assets / "inspector.urdf").write_text(robot_xml)


10

In [ ]:
layout = [
    ("west_barrier", "barrier", "obstacle", [-2.4, 0, 0.35]),
    ("east_barrier", "barrier", "obstacle", [2.4, 0, 0.35]),
    ("shelf_A", "shelf", "obstacle", [-1.3, 0.8, 0.45]),
    ("shelf_B", "shelf", "obstacle", [1.3, 0.8, 0.45]),
    ("inspection_A", "inspection_zone", "target", [-1.3, -0.8, 0.011]),
    ("inspection_B", "inspection_zone", "target", [1.3, -0.8, 0.011]),
    ("home_zone", "inspection_zone", "home", [0, -1.8, 0.011]),
    ("reference_A", "landmark", "landmark", [-2.0, 1.8, 0.5]),
    ("reference_B", "landmark", "landmark", [2.0, 1.8, 0.5]),
]
semantics = {}
for name, model, role, position in layout:
    body = p.loadURDF(models[model], position, useFixedBase=True)
    semantics[body] = dict(name=name, model=model, role=role)
robot = p.loadURDF(str(assets / "inspector.urdf"), [0, -1.8, 0.155])
print(json.dumps(semantics, indent=2))

def nearest_object(role):
    robot_xy = np.array(p.getBasePositionAndOrientation(robot)[0][:2])
    candidates = [(float(np.linalg.norm(np.array(p.getBasePositionAndOrientation(body)[0][:2]) - robot_xy)), body)
                  for body, metadata in semantics.items() if metadata["role"] == role]
    if not candidates: return None
    distance, body = min(candidates)
    return {**semantics[body], "body_id": body, "distance_m": round(distance, 3)}
print("Nearest inspection target:", nearest_object("target"))


error: Cannot load URDF file.

In [ ]:

{
  "nbformat": 4,
  "nbformat_minor": 5,
  "metadata": {
    "kernelspec": {
      "name": "python3",
      "language": "python",
      "display_name": "Python 3"
    }
  },
  "cells": [
    {
      "id": "60148b41",
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# ROS2 mission runbook\n",
        "\n",
        "Run these commands in separate TheConstruct shell windows. This notebook is documentation; it has not been executed in a Rosject.\n",
        "\n",
        "## Task 2: TheConstruct setup\n",
        "\n",
        "Target platform: **ROS2 Humble with Gazebo Classic**, not ROS1 Noetic or modern Gazebo/ros-gz. Choose a Rosject with the ROS2 TurtleBot3 Gazebo packages installed. Use the proposed name in ROSJECT_NAME.txt and verify it is available. This name is not an existing deployed project.\n",
        "\n",
        "Upload/extract this package into /home/user/rsp_project and open Task2_Runbook.ipynb inside the Rosject. The commands below assume that exact directory. If the extracted folder differs, adjust that one path.\n",
        "\n",
        "### Terminal 1: dependency preflight and build\n",
        "\n",
        "```bash\n",
        "source /opt/ros/humble/setup.bash\n",
        "ros2 pkg prefix gazebo_ros\n",
        "ros2 pkg prefix turtlebot3_gazebo\n",
        "cd /home/user/rsp_project/ros2_ws\n",
        "colcon build --symlink-install --packages-select warehouse_inspector\n",
        "source install/setup.bash\n",
        "ros2 launch warehouse_inspector simulation.launch.py\n",
        "```\n",
        "\n",
        "Keep Terminal 1 open. Open Gazebo's graphical window and confirm a Burger robot at the home marker, two inspection markers, shelves, barriers and landmarks. The launch file uses the installed TurtleBot3 Burger SDF. If preflight fails, use a matching Rosject; these instructions do not install or repair the hosted environment automatically. Do not launch two copies of the simulation.\n",
        "\n",
        "### Terminal 2: verify interfaces, then launch the controller\n",
        "\n",
        "```bash\n",
        "source /opt/ros/humble/setup.bash\n",
        "source /home/user/rsp_project/ros2_ws/install/setup.bash\n",
        "ros2 topic list -t\n",
        "ros2 topic info /scan --verbose\n",
        "ros2 topic info /odom --verbose\n",
        "ros2 topic info /cmd_vel --verbose\n",
        "ros2 launch warehouse_inspector mission.launch.py\n",
        "```\n",
        "\n",
        "Required message types are sensor_msgs/msg/LaserScan, nav_msgs/msg/Odometry and geometry_msgs/msg/Twist respectively. Names can be changed in config/mission.yaml. A TwistStamped interface requires an adapter; do not use this node unchanged in that environment. Only this controller should command velocity during a mission. Sensor subscribers use the sensor-data QoS profile.\n",
        "\n",
        "### Terminal 3: keyboard interface\n",
        "\n",
        "```bash\n",
        "source /opt/ros/humble/setup.bash\n",
        "source /home/user/rsp_project/ros2_ws/install/setup.bash\n",
        "ros2 run warehouse_inspector keyboard\n",
        "```\n",
        "\n",
        "Type `start` and press Enter. Other commands are `pause`, `resume`, `cancel`, `reset`, and `quit`. The keyboard is a line-based interface so it works in browser terminals. `quit` sends cancel. Start requires fresh, valid scan and odometry. A FAULT needs `reset` followed by `start`; it never resumes automatically.\n",
        "\n",
        "### Terminal 4: monitor and capture evidence\n",
        "\n",
        "```bash\n",
        "source /opt/ros/humble/setup.bash\n",
        "source /home/user/rsp_project/ros2_ws/install/setup.bash\n",
        "ros2 topic echo /inspection/status\n",
        "```\n",
        "\n",
        "CSV logs are written to /home/user/inspection_evidence. Save screenshots of IDLE, navigation, inspection, avoidance and COMPLETE with the relevant terminal visible. Export the actual CSV and copy it into evidence/theconstruct; do not replace it with the offline test CSV.\n",
        "\n",
        "### Required execution checks\n",
        "\n",
        "- Normal mission: two inspection points, then home; expect COMPLETE. Inspect the trajectory and CSV.\n",
        "- Pause/resume: pause while moving, verify zero velocity command, then resume.\n",
        "- Cancel: cancel while moving, verify IDLE and zero velocity command.\n",
        "- Obstacle: use Gazebo's object insertion tool to place a box ahead. Verify the stop and AVOID transition. Record recovery or FAULT honestly. Capture the box position.\n",
        "- Stale sensors: pause Gazebo physics via its UI while the mission is active. The wall-clock watchdog should stop commands and enter FAULT after the sensor timeout. Unpause, reset and rerun. Availability of pause services varies by environment.\n",
        "- Unreachable goal: select a blocked goal and record avoidance or mission timeout. Do not claim a global planner will find an alternative route.\n",
        "\n"
      ]
    }
  ]
}